In [ ]:
torch.matmul(input, other, out=None)

| 特性   | 说明                  |
| ---- | ------------------- |
| 操作   | 矩阵乘法（非逐元素相乘）        |
| 支持维度 | 1D, 2D, 3D+（批量矩阵乘法） |
| 广播   | 支持自动广播              |
| 等价符号 | `@` 运算符             |


### 1D × 1D（向量点积）

In [3]:
import torch

a = torch.randn(3)  # [3]
b = torch.randn(3)  # [3]
print(a,b)
result = torch.matmul(a, b)  # 标量
# 等价: a.dot(b) 或 (a * b).sum()
print(result)

print(f"{a.shape} @ {b.shape} = {result.shape}")
# torch.Size([]) 标量

tensor([-0.9123,  0.7631, -2.0041]) tensor([ 0.0912, -1.1070, -0.7393])
tensor(0.5538)
torch.Size([3]) @ torch.Size([3]) = torch.Size([])


### 2D × 2D（标准矩阵乘法）

In [7]:
A = torch.randn(2, 3)  # [2, 3]
B = torch.randn(3, 4)  # [3, 4]

print(A,"\n",B)
result = torch.matmul(A, B)  # [2, 4]
# 规则: (m, n) @ (n, p) = (m, p)
print(result)
print(f"{A.shape} @ {B.shape} = {result.shape}")
# torch.Size([2, 4])

tensor([[ 1.3479,  1.9173,  0.0216],
        [ 0.4307,  1.0134, -0.0422]]) 
 tensor([[ 0.5996,  1.1129, -0.5614,  2.1400],
        [-0.7930,  0.6353, -1.1455, -0.2054],
        [-1.6516,  0.6403, -1.0894,  0.2571]])
tensor([[-0.7478,  2.7320, -2.9764,  2.4962],
        [-0.4757,  1.0961, -1.3566,  0.7026]])
torch.Size([2, 3]) @ torch.Size([3, 4]) = torch.Size([2, 4])


In [ ]:
A: [2, 3]     B: [3, 4]      Result: [2, 4]

[a11 a12 a13]   [b11 b12 b13 b14]   [c11 c12 c13 c14]
[a21 a22 a23] @ [b21 b22 b23 b24] = [c21 c22 c23 c24]
                [b31 b32 b33 b34]

其中 c11 = a11*b11 + a12*b21 + a13*b31

### 1D × 2D（向量 × 矩阵）

In [8]:
a = torch.randn(3)     # [3]
B = torch.randn(3, 4)  # [3, 4]

result = torch.matmul(a, B)  # [4]
print(result)
# 1D向量视为行向量 [1, 3]，结果去掉维度1

print(f"{a.shape} @ {B.shape} = {result.shape}")
# torch.Size([4])

tensor([ 1.1609, -2.4060, -1.9091, -0.6595])
torch.Size([3]) @ torch.Size([3, 4]) = torch.Size([4])


### 2D × 1D（矩阵 × 向量）

In [ ]:
A = torch.randn(2, 3)  # [2, 3]
b = torch.randn(3)     # [3]

result = torch.matmul(A, b)  # [2]
# 1D向量视为列向量 [3, 1]，结果去掉维度1

print(f"{A.shape} @ {b.shape} = {result.shape}")
# torch.Size([2])

### 3D+ × 3D+（批量矩阵乘法）⭐重点

In [10]:
# 批量矩阵乘法（Batched Matrix Multiplication）
A = torch.randn(2, 3, 4)  # [batch=2, m=3, n=4]
B = torch.randn(2, 4, 5)  # [batch=2, n=4, p=5]
print(A)
print(B)

result = torch.matmul(A, B)  # [2, 3, 5]
print(result)
# 最后两维做矩阵乘法，前面维度广播

print(f"{A.shape} @ {B.shape} = {result.shape}")
# torch.Size([2, 3, 5])

tensor([[[-1.4638,  0.9264, -0.2532,  1.6818],
         [ 0.8823,  1.8891,  1.2996, -1.3582],
         [-0.7900, -1.5002,  1.6155, -0.3499]],

        [[ 0.4261, -1.9014, -0.7227,  0.0722],
         [-0.2813, -1.3824, -1.0456,  0.3128],
         [ 0.5885, -3.3770,  0.7761,  1.2453]]])
tensor([[[-0.9454, -1.3221,  0.0899, -0.3188, -0.2925],
         [ 1.1451,  1.1852, -1.3291,  1.5811, -0.1615],
         [ 2.8718, -1.4335, -1.3802, -0.6788,  0.6649],
         [-0.4145, -1.3047,  0.3317, -1.0615,  0.2627]],

        [[-0.6733,  1.6305,  1.5637,  1.3332,  0.0085],
         [-0.4679, -0.9535, -0.3198,  0.5825,  0.9405],
         [-1.3464, -0.1931,  1.9845, -1.4111,  1.1464],
         [ 0.1705, -0.5931, -2.4592,  0.4371,  0.4801]]])
tensor([[[ 1.0206,  1.2020, -0.4556,  0.3180,  0.5520],
         [ 5.6242,  0.9815, -4.6758,  3.2651, -0.0559],
         [ 3.8134, -2.5929, -0.4228, -2.8454,  1.4555]],

        [[ 1.5881,  2.6046, -0.3374,  0.5119, -2.5784],
         [ 2.2974,  0.8758, -2.8421,

In [11]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.k_dim = d_model // num_heads
        
    def forward(self, Q, K, V, mask=None):
        """
        Q, K, V: [batch, num_heads, seq_len, k_dim]
        """
        # ========== 第1次 matmul: Q @ K^T ==========
        # Q:     [batch, heads, seq_q, k_dim]
        # K^T:   [batch, heads, k_dim, seq_k]
        # 结果:  [batch, heads, seq_q, seq_k]
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.k_dim)
        print(f"Q @ K^T: {Q.shape} @ {K.transpose(-2,-1).shape} = {scores.shape}")
        # [2, 8, 100, 64] @ [2, 8, 64, 100] = [2, 8, 100, 100]
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = torch.softmax(scores, dim=-1)
        
        # ========== 第2次 matmul: attn @ V ==========
        # attn:  [batch, heads, seq_q, seq_k]
        # V:     [batch, heads, seq_k, k_dim]
        # 结果:  [batch, heads, seq_q, k_dim]
        
        output = torch.matmul(attn, V)
        print(f"attn @ V: {attn.shape} @ {V.shape} = {output.shape}")
        # [2, 8, 100, 100] @ [2, 8, 100, 64] = [2, 8, 100, 64]
        
        return output


# ========== 测试 ==========
mha = MultiHeadAttention()

# 模拟多头分割后的输入
batch, heads, seq, k_dim = 2, 8, 100, 64
Q = torch.randn(batch, heads, seq, k_dim)
K = torch.randn(batch, heads, seq, k_dim)
V = torch.randn(batch, heads, seq, k_dim)

output = mha(Q, K, V)

Q @ K^T: torch.Size([2, 8, 100, 64]) @ torch.Size([2, 8, 64, 100]) = torch.Size([2, 8, 100, 100])
attn @ V: torch.Size([2, 8, 100, 100]) @ torch.Size([2, 8, 100, 64]) = torch.Size([2, 8, 100, 64])
